<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH13/CH13_NB02_model_M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

# CH13 — 13.4 Building M: pruning and recovering the specialist
## Cascaded depth + width pruning with knowledge distillation

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
* **Author:** Pere Martra
* **Models:** Qwen3-0.6B (teacher **T**) → Model **M**
* **Colab Environment:** A100 GPU
* **Keys:**
  * Depth pruning
  * Width pruning
  * Knowledge distillation
  * optipfair
* **References:**
  * [Manning Book: Rearchitecting LLMs](https://livebook.manning.com/book/rearchitecting-llms)
  * [optipfair](https://github.com/peremartra/optipfair)
_____

Model **M** is the first step of a cascaded pruning-and-distillation family,
derived from a Qwen3-0.6B function-calling specialist (**T**).

| | T (teacher) | M (this notebook) |
|---|---|---|
| Layers | 28 | 25 |
| MLP intermediate | 3072 | 2304 |
| Expansion rate | 300% | 225% |
| Non-embedding params | measured below | measured below |

The pipeline is **depth pruning → width pruning → knowledge distillation**,
in that order. Depth is ranked on the intact teacher, then width is applied to
the depth-pruned model. Calibration prompts are reused for both pruning stages,
while the distillation teacher remains the unpruned model **T**.

Model S will be derived from M in a separate notebook, using M as its teacher.

## Setup

In [1]:
!pip install -q \
    transformers==5.18.0 \
    datasets==4.0.0 \
    accelerate==1.12.0 \
    scikit-learn \
    pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 16.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [2]:
!pip install git+https://github.com/peremartra/optipfair.git

  Cloning https://github.com/peremartra/optipfair.git to /tmp/pip-req-build-0ir5smuz
  Running command git clone --filter=blob:none --quiet https://github.com/peremartra/optipfair.git /tmp/pip-req-build-0ir5smuz
  Resolved https://github.com/peremartra/optipfair.git to commit eff530b3f534fe0f5071f934fda74e46350b2f16
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for optipfair: filename=optipfair-0.4.2-py3-none-any.whl size=65899 sha256=584d16d41e34e6bf105c0648776ba332c937d71ff48438820bbd2e6f5e9e9ceb
  Stored in directory: /tmp/pip-ephem-wheel-cache-pvnnulre/wheels/fa/47/75/39bd78f0b3b1b083d24dfce8ef60d4664225a8755a906716ae
Successfully built optipfair


In [3]:
import json
import re
import gc
import copy
import random
from collections import Counter

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

import optipfair as opf

print(torch.cuda.get_device_name(0))

NVIDIA A100-SXM4-40GB


In [4]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(42)

## Configuration

The values below define this run: a 30% stratified test split, 1152-token
calibration and training limits, three layers removed at depth, and a 225%
MLP expansion target. The KD loss uses hard labels plus teacher-logit KL
(`alpha=0.3`, `beta=0.7`); feature losses are disabled (`gamma=0`, `delta=0`).

The three distillation runs use seeds 43, 44, and 42. Seed 42 is selected in
advance as the canonical Model M run; the test scores are not used to choose
the checkpoint.

In [5]:
DATASET_NAME = "Salesforce/xlam-function-calling-60k"
TEACHER_REPO_ID = "oopere/qwen3-0.6b-weather-geo-specialist"
MODEL_M_REPO_ID = "oopere/qwen3-0.6b-weather-geo-M"

# --- Data ---
TEST_SIZE = 0.30
RANDOM_STATE = 42
MIN_EXAMPLES_PER_FUNCTION = 3
ALLOWED_FUNCTIONS = {"get_ip_zipcode", "get_city_from_zipcode", "local_weather_api"}

# --- Calibration (shared by depth ranking and width pruning) ---
CALIBRATION_BATCH_SIZE = 8
MAX_CALIBRATION_LENGTH = 1152

# --- Depth pruning ---
NUM_LAYERS_TO_REMOVE = 3          # 28 -> 25

# --- Width pruning ---
# Expansion rate is intermediate/hidden as a percentage. Hidden stays 1024
# throughout the cascade, so 225% means 2304 -- not 225% of anything else.
EXPANSION_RATE = 225              # 3072 -> 2304
EXPANSION_DIVISOR = 128           # 2304 = 18 x 128, so no rounding surprise

# --- Knowledge distillation ---
ALPHA = 0.3                       # hard labels (task CE)
BETA = 0.7                        # soft labels (KL vs teacher logits)
GAMMA = 0.0                       # feature alignment (off)
DELTA = 0.0                       # feature dynamics (off)
TEMPERATURE = 1.1
SKEW_ALPHA = 0.15
EPOCHS = 12
LEARNING_RATE = 3e-5
DISTILL_BATCH_SIZE = 4
ACCUMULATION_STEPS = 4            # effective batch = 16
MAX_TRAIN_LENGTH = 1152

SEEDS = [42, 43, 44]
CANONICAL_SEED = 42

PUSH_TO_HUB = True

##  Dataset (recap of 13.2)

Three functions, no extractability filter. Deduplication happens **before** the
split so test scores measure generalisation rather than memorisation.

One caveat worth keeping in mind when reading the numbers below: `normalize_query`
strips punctuation and lowercases but keeps digits, so two queries that differ
only in the IP address survive as distinct examples. Train and test therefore
share query templates with different values, which makes the task closer to
"copy this value into that field" than to recall. That is exactly the kind of
capability that survives pruning well.

In [6]:
def curate_domain_dataset(dataset, allowed_functions=ALLOWED_FUNCTIONS,
                          min_examples=MIN_EXAMPLES_PER_FUNCTION):
    matched = [
        ex for ex in dataset
        if len(json.loads(ex["answers"])) == 1
        and json.loads(ex["answers"])[0]["name"] in allowed_functions
    ]
    function_counts = Counter(json.loads(ex["answers"])[0]["name"] for ex in matched)
    main_functions = {name for name, count in function_counts.items() if count >= min_examples}
    return [ex for ex in matched if json.loads(ex["answers"])[0]["name"] in main_functions]

raw_dataset = load_dataset(DATASET_NAME, split="train")
domain_examples = curate_domain_dataset(raw_dataset)
print(f"Curated domain examples: {len(domain_examples)}")
for name, count in Counter(
    json.loads(ex["answers"])[0]["name"] for ex in domain_examples
).most_common():
    print(f"  {count:4d}  {name}")

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

xlam_function_calling_60k.json: reconstructing file:   0%|          |  0.00B / 96.1MB            

xlam_function_calling_60k.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Curated domain examples: 233
   125  get_ip_zipcode
    75  get_city_from_zipcode
    33  local_weather_api


In [7]:
def normalize_query(text):
    return re.sub(r"[^\w\s]", "", text.lower()).strip()

def deduplicate_by_query(examples):
    seen = set()
    unique = []
    for ex in examples:
        key = normalize_query(ex["query"])
        if key not in seen:
            seen.add(key)
            unique.append(ex)
    return unique

domain_examples = deduplicate_by_query(domain_examples)
print(f"After deduplication: {len(domain_examples)}")

After deduplication: 177


In [8]:
tokenizer = AutoTokenizer.from_pretrained(TEACHER_REPO_ID)
tokenizer.padding_side = "right"
print("pad_token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("eos_token:", repr(tokenizer.eos_token), tokenizer.eos_token_id)

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

pad_token: '<|endoftext|>' 151643
eos_token: '<|im_end|>' 151645


In [9]:
# Same prompt/completion split used to train the specialist -- completion is
# only the assistant's tool_call, matching completion_only_loss=True upstream.
def build_prompt_completion(example, tokenizer):
    tools = json.loads(example["tools"])
    answers = json.loads(example["answers"])
    user_message = {"role": "user", "content": example["query"]}
    assistant_message = {
        "role": "assistant",
        "content": None,
        "tool_calls": [{
            "type": "function",
            "function": {"name": answers[0]["name"], "arguments": answers[0]["arguments"]},
        }],
    }
    full_text = tokenizer.apply_chat_template(
        [user_message, assistant_message], tools=tools, tokenize=False, enable_thinking=False,
    )
    prompt_text = tokenizer.apply_chat_template(
        [user_message], tools=tools, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    assert full_text.startswith(prompt_text), "prompt is not a prefix of the full rendered text"
    return {"prompt": prompt_text, "completion": full_text[len(prompt_text):]}

function_labels = [json.loads(ex["answers"])[0]["name"] for ex in domain_examples]
train_examples, test_examples = train_test_split(
    domain_examples, test_size=TEST_SIZE, stratify=function_labels,
    random_state=RANDOM_STATE,
)
print(f"Train examples: {len(train_examples)}")
print(f"Test examples:  {len(test_examples)}")

train_dataset = Dataset.from_list([build_prompt_completion(ex, tokenizer) for ex in train_examples])
test_dataset = Dataset.from_list([build_prompt_completion(ex, tokenizer) for ex in test_examples])

Train examples: 123
Test examples:  54


In [10]:
# Sequence length budget. MAX_TRAIN_LENGTH truncates from the right, so any
# example longer than it loses the tail of its tool_call -- check how many.
lengths = [
    len(tokenizer(ex["prompt"] + ex["completion"])["input_ids"])
    for ex in train_dataset
]
print(f"train prompt+completion -- min {min(lengths)}, mean {sum(lengths)//len(lengths)}, max {max(lengths)}")
over = sum(1 for l in lengths if l > MAX_TRAIN_LENGTH)
print(f"over MAX_TRAIN_LENGTH ({MAX_TRAIN_LENGTH}): {over} ({over/len(lengths)*100:.1f}%)")

train prompt+completion -- min 193, mean 395, max 1110
over MAX_TRAIN_LENGTH (1152): 0 (0.0%)


In [11]:
def assert_tied(model):
    # embed_tokens and lm_head must share storage, as in Qwen3-0.6B. Checked on
    # the tensors in memory, never on the config (see the LoRA notebook).
    emb = model.get_input_embeddings().weight
    head = model.get_output_embeddings().weight
    assert emb.data_ptr() == head.data_ptr(), (
        "embed_tokens and lm_head are not tied -- check the transformers version"
    )

gc.collect()
torch.cuda.empty_cache()

teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_REPO_ID, dtype=torch.bfloat16,
).to("cuda")
assert_tied(teacher_model)
teacher_model.generation_config.pad_token_id = tokenizer.pad_token_id
teacher_model.eval()

print(f"teacher_model ready -- {teacher_model.config.num_hidden_layers} layers, "
      f"intermediate {teacher_model.config.intermediate_size}, frozen, bf16.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

teacher_model ready -- 28 layers, intermediate 3072, frozen, bf16.


## Teacher (T)

T is loaded as the unpruned specialist and used as the reference model for
baseline evaluation and knowledge distillation. The distillation routine
handles teacher freezing internally; this notebook does not modify T's
parameters before pruning copies are created.

`assert_tied` confirms that T's input embeddings and `lm_head` share one
tensor. Every pruned copy inherits that sharing, and the notebook checks it
again after each pruning stage and after distillation.

In [12]:
def generate_tool_call(model, tokenizer, example, max_new_tokens=200):
    tools = json.loads(example["tools"])
    messages = [{"role": "user", "content": example["query"]}]
    prompt = tokenizer.apply_chat_template(
        messages, tools=tools, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def parse_tool_call(generated_text):
    match = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", generated_text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError:
        return None

In [13]:
def is_substring_equivalent(a, b):
    a_clean = re.sub(r"[^\w\s]", "", str(a).lower()).strip()
    b_clean = re.sub(r"[^\w\s]", "", str(b).lower()).strip()
    if not a_clean or not b_clean:
        return False
    # Free-text values only. If either side has no letters after cleaning, bail
    # out: digit-substring collisions would otherwise produce false positives.
    if not re.search(r"[a-z]", a_clean) or not re.search(r"[a-z]", b_clean):
        return False
    # Require a minimum length so short strings do not match by sheer chance
    # (e.g. "i" being a substring of "imperial").
    if len(a_clean) < 3 or len(b_clean) < 3:
        return a_clean == b_clean
    return a_clean in b_clean or b_clean in a_clean

def is_close_coordinate(key, predicted, expected, tolerance_degrees=0.05):
    if not any(k in key.lower() for k in ["lat", "lon", "lng"]):
        return False
    try:
        return abs(float(predicted) - float(expected)) <= tolerance_degrees
    except (TypeError, ValueError):
        return False

def value_matches(key, predicted, expected, tolerant):
    if tolerant:
        try:
            if float(predicted) == float(expected):
                return True
        except (TypeError, ValueError):
            pass
        if isinstance(predicted, str) and isinstance(expected, str):
            if predicted.strip().lower() == expected.strip().lower():
                return True
            if not re.search(r"[a-z]", (predicted + expected).lower()):
                if re.sub(r"\s+", "", predicted) == re.sub(r"\s+", "", expected):
                    return True
            if is_substring_equivalent(predicted, expected):
                return True
        if is_close_coordinate(key, predicted, expected):
            return True
    return predicted == expected

def call_matches(predicted, ground_truth, tolerant):
    if predicted is None or predicted.get("name") != ground_truth["name"]:
        return False
    predicted_args = predicted.get("arguments", {})
    expected_args = ground_truth["arguments"]
    if not tolerant and set(predicted_args.keys()) != set(expected_args.keys()):
        return False
    return all(
        key in predicted_args and value_matches(key, predicted_args[key], expected_value, tolerant)
        for key, expected_value in expected_args.items()
    )

In [14]:
def evaluate_model(model, tokenizer, examples, verbose=True):
    """Evaluate and keep what was generated.

    Each result carries the raw output and the parsed call, so downstream
    analysis reads the very same generations these metrics were computed from --
    no regeneration, and no assumption that a second pass would produce
    identical text.
    """
    results = []
    for example in examples:
        ground_truth = json.loads(example["answers"])[0]
        generated_text = generate_tool_call(model, tokenizer, example)
        predicted = parse_tool_call(generated_text)
        results.append({
            "generated_text": generated_text,
            "predicted": predicted,
            "ground_truth": ground_truth,
            "valid_json": predicted is not None,
            "exact_match": call_matches(predicted, ground_truth, tolerant=False),
            "tolerant_match": call_matches(predicted, ground_truth, tolerant=True),
        })
    n = len(results)
    if verbose:
        print(f"  Valid JSON:      {sum(r['valid_json'] for r in results) / n:.1%}")
        print(f"  Exact match:     {sum(r['exact_match'] for r in results) / n:.1%}")
        print(f"  Tolerant match:  {sum(r['tolerant_match'] for r in results) / n:.1%}")
    return results

def breakdown_mismatches(results, verbose=True):
    no_call = wrong_function = wrong_args = 0
    for result in results:
        if result["tolerant_match"]:
            continue
        predicted = result["predicted"]
        if predicted is None:
            no_call += 1
        elif predicted.get("name") != result["ground_truth"]["name"]:
            wrong_function += 1
        else:
            wrong_args += 1
    if verbose:
        print(f"  No tool_call emitted:        {no_call}")
        print(f"  Wrong function selected:     {wrong_function}")
        print(f"  Right function, wrong args:  {wrong_args}")
    return {"no_call": no_call, "wrong_function": wrong_function, "wrong_args": wrong_args}

In [15]:
def count_parameters(model):
    # Counts what is in memory. model.parameters() yields a shared tensor once,
    # so a tied lm_head is not counted twice; an untied one is reported as part
    # of the embeddings. The config flag is not consulted: it can disagree with
    # the weights actually loaded.
    total = sum(p.numel() for p in model.parameters())
    emb = model.get_input_embeddings().weight
    head = model.get_output_embeddings().weight
    vocab = emb.numel() + (0 if head.data_ptr() == emb.data_ptr() else head.numel())
    return {"total": total, "embeddings": vocab, "non_embedding": total - vocab}

teacher_params = count_parameters(teacher_model)
print({k: f"{v/1e6:.1f}M" for k, v in teacher_params.items()})

{'total': '596.0M', 'embeddings': '155.6M', 'non_embedding': '440.5M'}


In [16]:
print("T (specialist, unpruned) -- baseline on test_examples:")
teacher_results = evaluate_model(teacher_model, tokenizer, test_examples)
teacher_breakdown = breakdown_mismatches(teacher_results)

T (specialist, unpruned) -- baseline on test_examples:
  Valid JSON:      100.0%
  Exact match:     96.3%
  Tolerant match:  96.3%
  No tool_call emitted:        0
  Wrong function selected:     0
  Right function, wrong args:  2


## 13.4.1 Pruning the specialist

The same dataloader serves both pruning stages. `prune_model` runs the forward
pass internally on whichever model it is given, so passing this dataloader to
the depth-pruned model automatically measures activations on the 25-layer
topology — no separate recalibration step is needed.

### Calibration data

In [17]:
CALIBRATION_BATCH_SIZE = 8
MAX_CALIBRATION_LENGTH = 1152 #A

calibration_texts = list(train_dataset["prompt"]) #B
lens = [len(tokenizer(t)["input_ids"])
        for t in calibration_texts]
print(f"calibration token lengths -- min {min(lens)}, "
      f"mean {sum(lens)//len(lens)}, max {max(lens)}")
assert max(lens) <= MAX_CALIBRATION_LENGTH, ( #1
    f"truncation would drop part of the prompt: "
    f"raise MAX_CALIBRATION_LENGTH to >= {max(lens)}"
)

calibration_dataset = Dataset.from_dict({"text": calibration_texts})
calibration_dataset = calibration_dataset.map(
    lambda ex: tokenizer(
        ex["text"], truncation=True,
        max_length=MAX_CALIBRATION_LENGTH, return_tensors=None,
    ),
    batched=True,
)

def collate_calibration(batch): #2
    max_len = max(len(ex["input_ids"]) for ex in batch)
    pad_id = tokenizer.pad_token_id
    return {
        "input_ids": torch.tensor([
            ex["input_ids"] + [pad_id] * (max_len - len(ex["input_ids"]))
            for ex in batch
        ]),
        "attention_mask": torch.tensor([
            ex["attention_mask"]
            + [0] * (max_len - len(ex["attention_mask"]))
            for ex in batch
        ]),
    }

calibration_dataloader = DataLoader(
    calibration_dataset, batch_size=CALIBRATION_BATCH_SIZE,
    collate_fn=collate_calibration,
)

calibration token lengths -- min 165, mean 363, max 1074


Map:   0%|          | 0/123 [00:00<?, ? examples/s]

## Depth pruning: T → 25 blocks

Layer importance is scored by cosine distance between each block's input and
output. The selector then scans the ascending importance ranking and removes
three layers while protecting layers 0-3 and the final two layers and avoiding
adjacent removals.

The resulting model is evaluated immediately after depth pruning. Its surviving
layers are also recorded in `depth_index_map`, because width pruning and the
next notebook refer to the post-depth numbering.

In [18]:
importance_scores = opf.analyze_layer_importance(
    teacher_model, calibration_dataloader, show_progress=True
)
sorted_layers = sorted(importance_scores.items(), key=lambda x: x[1])

print("Layers by importance (ascending -- most passive first):")
for layer_idx, score in sorted_layers:
    print(f"  Layer {layer_idx:2d}: {score:.6f}")

Processing batches: 100%|██████████| 16/16 [00:02<00:00,  7.37it/s]

Layers by importance (ascending -- most passive first):
  Layer 25: 0.030985
  Layer 26: 0.031850
  Layer 23: 0.038008
  Layer 24: 0.040039
  Layer 13: 0.043818
  Layer 12: 0.044896
  Layer 15: 0.053816
  Layer 14: 0.054917
  Layer 22: 0.060315
  Layer 18: 0.070633
  Layer 11: 0.071832
  Layer 17: 0.075268
  Layer 21: 0.075675
  Layer 16: 0.077887
  Layer 20: 0.078123
  Layer 19: 0.084242
  Layer 10: 0.086085
  Layer  8: 0.090849
  Layer  5: 0.092188
  Layer  9: 0.092877
  Layer  7: 0.095624
  Layer 27: 0.097521
  Layer  6: 0.107508
  Layer  4: 0.113115
  Layer  2: 0.116143
  Layer  3: 0.122403
  Layer  1: 0.153281
  Layer  0: 0.943267


In [19]:
def select_layers_to_prune(importance_scores, num_layers_to_remove,
                           heuristic_protection=True, adjacent_protection=True):
    num_layers = len(importance_scores)
    protected = {0, 1, 2, 3, num_layers - 2, num_layers - 1} if heuristic_protection else set()

    sorted_layers = sorted(importance_scores.items(), key=lambda x: x[1])
    selected = []
    for layer, score in sorted_layers:
        if heuristic_protection and layer in protected:
            continue
        if adjacent_protection and any(abs(layer - l) == 1 for l in selected):
            continue
        selected.append(layer)
        if len(selected) >= num_layers_to_remove:
            break

    return sorted(selected)

layers_to_remove = select_layers_to_prune(
    importance_scores, NUM_LAYERS_TO_REMOVE,
    heuristic_protection=True, adjacent_protection=True,
)
print(f"Layers selected for removal: {layers_to_remove}")

Layers selected for removal: [13, 23, 25]


In [20]:
depth_pruned_model, depth_stats = opf.prune_model(
    model=copy.deepcopy(teacher_model),
    pruning_type="DEPTH",
    layer_indices=layers_to_remove,
    show_progress=True,
    return_stats=True,
)
depth_pruned_model.generation_config.pad_token_id = tokenizer.pad_token_id
depth_pruned_model.eval()
assert_tied(depth_pruned_model)

print({k: depth_stats[k] for k in [
    "original_layer_count", "final_layer_count", "layers_removed",
    "layer_reduction_percentage", "percentage_reduction",
]})

Removing layers: 100%|██████████| 28/28 [00:00<00:00, 344400.33it/s]

{'original_layer_count': 28, 'final_layer_count': 25, 'layers_removed': 3, 'layer_reduction_percentage': 10.714285714285714, 'percentage_reduction': 7.917597237493128}


In [21]:
# Layer indices are renumbered after depth pruning. Keep the mapping: the
# width-pruning layer_indices below and anything referring back to T need it.
surviving = [i for i in range(teacher_model.config.num_hidden_layers)
             if i not in layers_to_remove]
depth_index_map = {new: old for new, old in enumerate(surviving)}
print("new -> original layer index:")
print(depth_index_map)

new -> original layer index:
{0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 14, 14: 15, 15: 16, 16: 17, 17: 18, 18: 19, 19: 20, 20: 21, 21: 22, 22: 24, 23: 26, 24: 27}


In [22]:
print(f"After depth pruning ({depth_pruned_model.config.num_hidden_layers} layers) "
      "-- on test_examples:")
post_depth_results = evaluate_model(depth_pruned_model, tokenizer, test_examples)
post_depth_breakdown = breakdown_mismatches(post_depth_results)

After depth pruning (25 layers) -- on test_examples:
  Valid JSON:      72.2%
  Exact match:     55.6%
  Tolerant match:  72.2%
  No tool_call emitted:        15
  Wrong function selected:     0
  Right function, wrong args:  0


## Width pruning: 3072 → 2304

Neuron importance uses PPM in hybrid mode: supplying the calibration
dataloader lets the ranking combine weight information with activation
statistics from tool-calling prompts. In `optipfair`, this mode is selected with
`neuron_selection_method="PPM"`.

The width target is configured as a 225% expansion over the 1024-wide hidden
state, with divisor 128, so the intended intermediate size is 2304. Width
pruning is applied to the depth-pruned model, and the following checks report
the actual per-layer sizes and parameter counts before distillation.

In [23]:
import copy

model_m_pruned, width_stats = opf.prune_model(
    model=copy.deepcopy(depth_pruned_model),
    pruning_type="MLP_GLU",
    neuron_selection_method="PPM",     # PPM; the only method supporting hybrid mode
    pruning_percentage=None,           # must be None when expansion_rate is set
    expansion_rate=EXPANSION_RATE,
    expansion_divisor=EXPANSION_DIVISOR,
    dataloader=calibration_dataloader, # enables the data-driven hybrid ranking
    show_progress=True,
    return_stats=True,
)
model_m_pruned.generation_config.pad_token_id = tokenizer.pad_token_id
model_m_pruned.eval()
assert_tied(model_m_pruned)

print({k: width_stats[k] for k in ["original_parameters", "pruned_parameters",
                                   "percentage_reduction"]})

Pruning all layers: 100%|██████████| 25/25 [00:01<00:00, 20.68it/s]

{'original_parameters': 548857088, 'pruned_parameters': 489874688, 'percentage_reduction': 10.746403989229341}


In [24]:
print(depth_pruned_model.model.layers[2].mlp.gate_proj.out_features)
print(model_m_pruned.model.layers[2].mlp.gate_proj.out_features)

3072
2304


In [25]:
# Confirm the architecture landed where intended.
inter = [layer.mlp.gate_proj.out_features for layer in model_m_pruned.model.layers]
print(f"layers: {len(inter)}")
print(f"intermediate sizes: {Counter(inter)}")

m_params = count_parameters(model_m_pruned)
print({k: f"{v/1e6:.1f}M" for k, v in m_params.items()})
print(f"non-embedding reduction vs T: "
      f"{1 - m_params['non_embedding'] / teacher_params['non_embedding']:.1%}")

layers: 25
intermediate sizes: Counter({2304: 25})
{'total': '489.9M', 'embeddings': '155.6M', 'non_embedding': '334.3M'}
non-embedding reduction vs T: 24.1%


In [26]:
print("After depth + width pruning, before KD -- on test_examples:")
pre_kd_results = evaluate_model(model_m_pruned, tokenizer, test_examples)
pre_kd_breakdown = breakdown_mismatches(pre_kd_results)

After depth + width pruning, before KD -- on test_examples:
  Valid JSON:      74.1%
  Exact match:     51.9%
  Tolerant match:  72.2%
  No tool_call emitted:        14
  Wrong function selected:     0
  Right function, wrong args:  1


## 13.4.2 Recovering knowledge

Teacher is **T**, the unpruned specialist. It is the only model in the family
that has never lost capability, so it provides the cleanest supervision.

Labels are masked to completion-only (`-100` on prompt and padding), so the
task loss scores the `<tool_call>` block rather than the reconstruction of the
tools JSON that dominates each sequence. This mirrors `completion_only_loss`
from the LoRA run upstream.


In [27]:
# Completion-only labels: prompt tokens and padding are masked to -100 so the
# task loss (and, when active, the feature losses) only score the <tool_call>
# the model is supposed to produce.
def build_distillation_example(prompt, completion, tokenizer, max_length=MAX_TRAIN_LENGTH):
    full_text = prompt + completion
    full_encoded = tokenizer(full_text, truncation=True, max_length=max_length)
    prompt_ids = tokenizer(prompt, truncation=True, max_length=max_length)["input_ids"]
    prompt_len = len(prompt_ids)

    labels = list(full_encoded["input_ids"])
    for i in range(min(prompt_len, len(labels))):
        labels[i] = -100

    return {
        "input_ids": full_encoded["input_ids"],
        "attention_mask": full_encoded["attention_mask"],
        "labels": labels,
    }

distillation_examples = [
    build_distillation_example(ex["prompt"], ex["completion"], tokenizer)
    for ex in train_dataset
]
distillation_dataset = Dataset.from_list(distillation_examples)

# Sanity check: at least one supervised token must survive the masking in
# every example, otherwise that example contributes nothing to the task loss.
unsupervised = [
    i for i, ex in enumerate(distillation_examples)
    if all(l == -100 for l in ex["labels"])
]
assert not unsupervised, f"examples with no supervised token: {unsupervised}"
print(f"{len(distillation_dataset)} distillation examples, all with supervised targets.")

123 distillation examples, all with supervised targets.


In [28]:
def collate_distillation(batch):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    input_ids, attention_mask, labels = [], [], []
    for ex in batch:
        pad_len = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(ex["attention_mask"] + [0] * pad_len)
        labels.append(ex["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attention_mask),
        "labels": torch.tensor(labels),
    }

distillation_dataloader = DataLoader(
    distillation_dataset, batch_size=DISTILL_BATCH_SIZE, shuffle=True,
    collate_fn=collate_distillation,
)

In [29]:
# Three seeds, same configuration. With 54 test examples each error is worth
# 1.85 points, so two examples shifting is a 4-point swing -- enough to cover
# most hyperparameter differences. Running the same configuration under several
# seeds is what tells a real effect from batch-order noise.


seed_runs = {}

for seed in SEEDS:
    print(f"\n{'=' * 60}\nseed {seed}\n{'=' * 60}")
    set_seed(seed)

    student = copy.deepcopy(model_m_pruned)
    distilled, stats = opf.distill_model(
        student_model=student,
        teacher_model=teacher_model,
        dataloader=distillation_dataloader,
        alpha=ALPHA,
        beta=BETA,
        gamma=GAMMA,
        delta=DELTA,
        temperature=TEMPERATURE,
        skew_alpha=SKEW_ALPHA,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        accumulation_steps=ACCUMULATION_STEPS,
        show_progress=True,
        return_stats=True,
    )
    distilled.generation_config.pad_token_id = tokenizer.pad_token_id
    distilled.eval()
    assert_tied(distilled)

    results = evaluate_model(distilled, tokenizer, test_examples, verbose=False)
    exact = sum(r["exact_match"] for r in results) / len(results)
    print(f"  exact match: {exact:.1%}")

    seed_runs[seed] = {"results": results, "stats": stats, "exact_match": exact}

    # Model M is the canonical seed's model, fixed in advance -- not the best of
    # the three. Picking the best run after seeing the scores would be selecting
    # on the test set.
    if seed == CANONICAL_SEED:
        model_m = distilled
        distill_stats = stats
    else:
        del distilled
        gc.collect()
        torch.cuda.empty_cache()


seed 42


Epoch 1/12:   0%|          | 0/31 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 2/12:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 3/12:   0%|          | 0/31 [00:00<?, ?it/s]

...

Epoch 9/12:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 10/12:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 11/12:   0%|          | 0/31 [00:00<?, ?it/s]

Epoch 12/12:   0%|          | 0/31 [00:00<?, ?it/s]

  exact match: 90.7%


In [30]:
scores = [seed_runs[s]["exact_match"] for s in SEEDS]
variance_df = pd.DataFrame([
    {"seed": s, "exact_match": f"{seed_runs[s]['exact_match']:.1%}"}
    for s in SEEDS
] + [
    {"seed": "range", "exact_match": f"{(max(scores)-min(scores))*100:.1f} pp"},
    {"seed": "range", "exact_match": f"{max(scores)-min(scores)*100:.1f} pp"},
])
print(f"Spread across seeds: {(max(scores)-min(scores))*len(test_examples):.0f} "
      f"of {len(test_examples)} test examples")
variance_df

Spread across seeds: 3 of 54 test examples


,seed,exact_match
0,42,92.6%
1,43,96.3%
2,44,90.7%
3,range,5.6 pp
4,range,-89.8 pp


In [31]:
loss_total = distill_stats["loss_history"]["total"]
loss_task = distill_stats["loss_history"]["task"]
loss_logits = distill_stats["loss_history"]["logits"]

print(f"{'epoch':>6}  {'total':>8}  {'task':>8}  {'logits':>8}")
for i, (t, ta, lo) in enumerate(zip(loss_total, loss_task, loss_logits), 1):
    print(f"{i:>6}  {t:>8.4f}  {ta:>8.4f}  {lo:>8.4f}")

best = min(range(len(loss_total)), key=lambda i: loss_total[i]) + 1
print(f"\nlowest total loss at epoch {best} of {len(loss_total)}")
if best < len(loss_total):
    print("Training loss rose after that point. With no validation split there is"
          " no way to tell overfitting from noise here -- worth watching.")

 epoch     total      task    logits
     1    0.0210    0.0612    0.0038
     2    0.0062    0.0191    0.0006
     3    0.0037    0.0116    0.0004
     4    0.0013    0.0038    0.0002
     5    0.0002    0.0007   -0.0000
     6    0.0001    0.0005   -0.0001
     7    0.0001    0.0004   -0.0000
     8    0.0001    0.0004   -0.0000
     9    0.0001    0.0004   -0.0000
    10    0.0001    0.0003   -0.0000
    11    0.0001    0.0004   -0.0000
    12    0.0001    0.0003   -0.0000

lowest total loss at epoch 12 of 12


## Results

In [32]:
# Reuse the canonical seed's generations rather than regenerating them.
post_kd_results = seed_runs[CANONICAL_SEED]["results"]
n = len(post_kd_results)
print(f"M (25 layers, 2304 intermediate, post-KD, seed {CANONICAL_SEED}) "
      "-- on test_examples:")
print(f"  Valid JSON:      {sum(r['valid_json'] for r in post_kd_results) / n:.1%}")
print(f"  Exact match:     {sum(r['exact_match'] for r in post_kd_results) / n:.1%}")
print(f"  Tolerant match:  {sum(r['tolerant_match'] for r in post_kd_results) / n:.1%}")
post_kd_breakdown = breakdown_mismatches(post_kd_results)

M (25 layers, 2304 intermediate, post-KD, seed 42) -- on test_examples:
  Valid JSON:      100.0%
  Exact match:     92.6%
  Tolerant match:  92.6%
  No tool_call emitted:        0
  Wrong function selected:     0
  Right function, wrong args:  4


In [33]:
def summarize(results, breakdown, label):
    n = len(results)
    return {
        "model": label,
        "valid_json": sum(r["valid_json"] for r in results) / n,
        "exact_match": sum(r["exact_match"] for r in results) / n,
        "tolerant_match": sum(r["tolerant_match"] for r in results) / n,
        "no_call": breakdown["no_call"],
        "wrong_function": breakdown["wrong_function"],
        "wrong_args": breakdown["wrong_args"],
    }

comparison_df = pd.DataFrame([
    summarize(teacher_results, teacher_breakdown, "T (28L, 3072)"),
    summarize(post_depth_results, post_depth_breakdown,
              f"depth only ({depth_pruned_model.config.num_hidden_layers}L, 3072)"),
    summarize(pre_kd_results, pre_kd_breakdown, "depth+width, pre-KD (25L, 2304)"),
    summarize(post_kd_results, post_kd_breakdown, "M, post-KD (25L, 2304)"),
])
comparison_df

,model,valid_json,exact_match,tolerant_match,no_call,wrong_function,wrong_args
0,"T (28L, 3072)",1.000000,0.962963,0.962963,0,0,2
1,"depth only (25L, 3072)",0.722222,0.555556,0.722222,15,0,0
2,"depth+width, pre-KD (25L, 2304)",0.740741,0.518519,0.722222,14,0,1
3,"M, post-KD (25L, 2304)",1.000000,0.925926,0.925926,0,0,4


In [34]:
# Per-function accuracy: an aggregate can hide one function collapsing while
# the others carry the average.
function_results = {}
for ex, result in zip(test_examples, post_kd_results):
    fn_name = json.loads(ex["answers"])[0]["name"]
    function_results.setdefault(fn_name, []).append(result["exact_match"])

for fn_name, matches in function_results.items():
    print(f"{fn_name}: {sum(matches)}/{len(matches)} exact match")

local_weather_api: 7/10 exact match
get_ip_zipcode: 25/26 exact match
get_city_from_zipcode: 18/18 exact match


In [35]:
# Are M's failures a subset of T's, or does M fail on cases T handled?
teacher_wrong = {i for i, r in enumerate(teacher_results) if not r["tolerant_match"]}
m_wrong = {i for i, r in enumerate(post_kd_results) if not r["tolerant_match"]}

print(f"T failures: {len(teacher_wrong)}   M failures: {len(m_wrong)}")
print(f"Extra failures vs teacher: {len(m_wrong - teacher_wrong)}")
for i in sorted(m_wrong - teacher_wrong):
    print("-" * 50)
    print("QUERY:       ", test_examples[i]["query"])
    print("GENERATED:   ", post_kd_results[i]["generated_text"])
    print("GROUND TRUTH:", post_kd_results[i]["ground_truth"])

T failures: 2   M failures: 4
Extra failures vs teacher: 2
--------------------------------------------------
QUERY:        Fetch the weather forecast for Los Angeles, CA for the next 5 days including air quality data in English.
GENERATED:    <tool_call>
{"name": "local_weather_api", "arguments": {"q": "Los Angeles, CA", "aqi": "yes", "num_of_days": 5}}
</tool_call>
GROUND TRUTH: {'name': 'local_weather_api', 'arguments': {'q': 'Los Angeles, CA', 'num_of_days': 5, 'aqi': 'yes', 'lang': 'en'}}
--------------------------------------------------
QUERY:        Could you provide the current weather conditions and a 5-day forecast for New York City, including air quality data but no weather alerts, in French?
GENERATED:    <tool_call>
{"name": "local_weather_api", "arguments": {"q": "New York City", "aqi": "yes", "lang": "fr", "num_of_days": 5}}
</tool_call>
GROUND TRUTH: {'name': 'local_weather_api', 'arguments': {'q': 'New York City', 'aqi': 'yes', 'lang': 'fr', 'num_of_days': 5, 'alerts'

In [36]:
# Bootstrap confidence intervals. With ~50 test examples a one- or two-example
# difference is well inside the noise, so report intervals rather than treating
# a point estimate as a ranking.
import numpy as np

def bootstrap_ci(flags, n_boot=10000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    flags = np.asarray(flags, dtype=float)
    means = flags[rng.integers(0, len(flags), size=(n_boot, len(flags)))].mean(axis=1)
    return flags.mean(), np.quantile(means, alpha / 2), np.quantile(means, 1 - alpha / 2)

for label, results in [
    ("T      ", teacher_results),
    ("pre-KD ", pre_kd_results),
    ("M      ", post_kd_results),
]:
    mean, lo, hi = bootstrap_ci([r["exact_match"] for r in results])
    print(f"{label} exact match: {mean:.1%}  95% CI [{lo:.1%}, {hi:.1%}]")

T       exact match: 96.3%  95% CI [90.7%, 100.0%]
pre-KD  exact match: 51.9%  95% CI [38.9%, 64.8%]
M       exact match: 92.6%  95% CI [85.2%, 98.1%]


In [37]:
family_df = pd.DataFrame([
    {
        "model": "T",
        "layers": teacher_model.config.num_hidden_layers,
        "intermediate": teacher_model.config.intermediate_size,
        "expansion_rate": f"{teacher_model.config.intermediate_size * 100 // teacher_model.config.hidden_size}%",
        "non_emb_params_M": round(teacher_params["non_embedding"] / 1e6, 1),
        "total_params_M": round(teacher_params["total"] / 1e6, 1),
        "emb_pct_of_total": f"{teacher_params['embeddings'] / teacher_params['total']:.0%}",
        "saved_dtype": "float16",   # trained with fp16 AMP on a T4 (no bf16 support)
        "kd_teacher": "-",
    },
    {
        "model": "M",
        "layers": model_m.config.num_hidden_layers,
        "intermediate": EXPANSION_RATE * model_m.config.hidden_size // 100,
        "expansion_rate": f"{EXPANSION_RATE}%",
        "non_emb_params_M": round(m_params["non_embedding"] / 1e6, 1),
        "total_params_M": round(m_params["total"] / 1e6, 1),
        "emb_pct_of_total": f"{m_params['embeddings'] / m_params['total']:.0%}",
        "saved_dtype": str(model_m.dtype).replace("torch.", ""),
        "kd_teacher": "T",
    },
])
family_df

,model,layers,intermediate,expansion_rate,non_emb_params_M,total_params_M,emb_pct_of_total,saved_dtype,kd_teacher
0,T,28,3072,300%,440.5,596.0,26%,float16,-
1,M,25,2304,225%,334.3,489.9,32%,bfloat16,T


The parameter table separates embeddings from the parameters affected by this
cascade. Depth pruning removes blocks and width pruning changes MLPs, while the
vocabulary matrix, shared by the input embeddings and the output head, is not
pruned here. Read the reported `non_embedding` count as the clearest measure of
the parameters removed by the architecture changes; the total count still
includes the unchanged embedding share, which grows as a fraction of the model
with every pruning step.

`saved_dtype` is the precision each checkpoint is stored in. T was trained on a
T4, which has no bf16 support, so it is stored in fp16. M is pruned and
distilled on an A100 in bf16 and is stored in bf16, the precision it was
trained in. Converting a checkpoint to another precision for inference is a
deployment decision, and it is verified where it is made.

## Saving model M

In [38]:
OUTPUT_DIR = "./model_m"
model_m.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

# Reload and verify the checkpoint stores a single vocabulary matrix.
reloaded = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR, dtype=torch.bfloat16)
assert_tied(reloaded)
assert count_parameters(reloaded)["total"] == m_params["total"]
print(f"Reloaded checkpoint: {count_parameters(reloaded)['total'] / 1e6:.1f}M parameters, "
      "embeddings tied")
del reloaded
gc.collect()

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()
    model_m.push_to_hub(MODEL_M_REPO_ID, private=True)
    tokenizer.push_to_hub(MODEL_M_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{MODEL_M_REPO_ID}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./model_m


Loading weights:   0%|          | 0/277 [00:00<?, ?it/s]

Reloaded checkpoint: 489.9M parameters, embeddings tied


README.md:   0%|          | 0.00/9.10k [00:00<?, ?B/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpmgvw656w/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to https://huggingface.co/oopere/qwen3-0.6b-weather-geo-M


In [39]:
# Carry forward to the S notebook: M becomes both the pruning source and the
# distillation teacher for the next step of the cascade.
handoff = {
    "model_m_repo": MODEL_M_REPO_ID,
    "layers": model_m.config.num_hidden_layers,
    "intermediate_size": EXPANSION_RATE * model_m.config.hidden_size // 100,
    "layers_removed_from_T": layers_to_remove,
    "depth_index_map": depth_index_map,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "m_exact_match": sum(r["exact_match"] for r in post_kd_results) / len(post_kd_results),
}
with open("model_m_handoff.json", "w") as f:
    json.dump(handoff, f, indent=2)
print(json.dumps(handoff, indent=2))

{
  "model_m_repo": "oopere/qwen3-0.6b-weather-geo-M",
  "layers": 25,
  "intermediate_size": 2304,
  "layers_removed_from_T": [
    13,
    23,
    25
  ],
  "depth_index_map": {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "10": 10,
    "11": 11,
    "12": 12,
    "13": 14,
    "14": 15,
    "15": 16,
    "16": 17,
    "17": 18,
    "18": 19,
    "19": 20,
    "20": 21,
    "21": 22,
    "22": 24,
    "23": 26,
    "24": 27
  },
  "random_state": 42,
  "test_size": 0.3,
  "m_exact_match": 0.9259259259259259
}


### Before moving on to S

S is pruned from M and distilled with M as its teacher, so everything downstream
inherits M's quality. Two things to settle first:

1. **Is M's accuracy acceptable?** A weak M means S starts from a degraded
   teacher and from a layer-importance ranking measured on a model that does
   not work well.
2. **Re-run the layer analysis on M.** Removing layers redistributes redundancy
   among the survivors, so the ranking computed on T does not carry over. The
   three layers to drop from M are not "the next three" from T's ranking.

Reuse the same split — `RANDOM_STATE` and `TEST_SIZE` are in the handoff file
for exactly that reason. The handoff also records M's architecture and the
mapping from M's post-depth indices back to T. Changing the dataset between
stages makes the family's numbers incomparable.